# Cylinder + Beam FSI — Mixed-Form PINN (TF2)

Steady 2D laminar flow past a **cylinder with an attached rigid beam**.
Based on the Turek & Hron (2006) CFD benchmark geometry.

> Turek, S. & Hron, J. (2006). *Proposal for Numerical Benchmarking of
> Fluid-Structure Interaction.* LNCSE.

**Key feature in this notebook:** each loss component has its own tunable
weight (`w_fu`, `w_fv`, `w_fs11`, `w_fs22`, `w_fs12`, `w_fp`,
`w_wall`, `w_inlet`, `w_outlet`), and every unweighted residual is
recorded and plotted separately after training.

## 1. Imports

In [ ]:
import numpy as np
import time
import pickle
import scipy.io
import scipy.optimize
import matplotlib.pyplot as plt
import tensorflow as tf

tf.random.set_seed(1234)
np.random.seed(1234)

## 2. Sampling & Geometry Helpers

In [ ]:
def lhs(n_dim, n_samples, seed=None):
    """Latin Hypercube Sampling in [0,1]^n_dim."""
    rng = np.random.default_rng(seed)
    cut = np.linspace(0.0, 1.0, n_samples + 1)
    u = rng.random((n_samples, n_dim))
    a, b = cut[:n_samples], cut[1:n_samples + 1]
    rdpoints = a[:, None] + (b - a)[:, None] * u
    H = np.empty_like(rdpoints)
    for j in range(n_dim):
        H[:, j] = rdpoints[rng.permutation(n_samples), j]
    return H


def DelObsPT(XY, xc, yc, r, xb0, xb1, yb0, yb1):
    """Remove points inside cylinder OR inside beam rectangle."""
    dst    = np.sqrt((XY[:,0]-xc)**2 + (XY[:,1]-yc)**2)
    in_cyl = dst <= r
    in_beam = ((XY[:,0] >= xb0) & (XY[:,0] <= xb1) &
               (XY[:,1] >= yb0) & (XY[:,1] <= yb1))
    return XY[~(in_cyl | in_beam), :]


def preprocess_mat(dir_path):
    """Load Fluent reference solution (.mat)."""
    data = scipy.io.loadmat(dir_path)
    X, Y = data['x'], data['y']
    vx, vy, P = data['vx'], data['vy'], data['p']
    return (X.flatten()[:,None], Y.flatten()[:,None],
            vx.flatten()[:,None], vy.flatten()[:,None],
            P.flatten()[:,None])

## 3. Model — MLP + Per-Term Weighted PINN

Each residual term has its own weight `w_*`.  
Loss histories are recorded as **unweighted** raw MSEs so you can
diagnose which physics equation is not yet satisfied, independent
of how the weights shift the total loss scale.

In [ ]:
DEFAULT_WEIGHTS = {
    'w_fu':     1.0,   # x-momentum residual
    'w_fv':     1.0,   # y-momentum residual
    'w_fs11':   1.0,   # constitutive: normal stress 11
    'w_fs22':   1.0,   # constitutive: normal stress 22
    'w_fs12':   1.0,   # constitutive: shear stress 12
    'w_fp':     1.0,   # trace / pressure-recovery constraint
    'w_wall':  10.0,   # no-slip BC (cylinder + beam + channel walls)
    'w_inlet': 10.0,   # prescribed inlet velocity
    'w_outlet': 10.0,  # zero mean pressure at outlet
}


class PINN_beam_flow_TF2:
    """Mixed-form PINN for steady laminar flow past cylinder + rigid beam.

    Network outputs (5 values): ψ, p, s11, s22, s12
      u = ∂ψ/∂y,   v = -∂ψ/∂x
    PDE residuals (Turek-Hron CFD benchmark, steady):
      f_u   = ρ(u u_x + v u_y) - s11_x - s12_y
      f_v   = ρ(u v_x + v v_y) - s12_x - s22_y
      f_s11 = -p + 2μ u_x - s11
      f_s22 = -p + 2μ v_y - s22
      f_s12 = μ(u_y + v_x) - s12
      f_p   = p + ½(s11 + s22)   [trace / pressure recovery]

    Each loss term has its own scalar weight set via the `weights` dict.
    All weights default to DEFAULT_WEIGHTS and can be partially overridden:

        model = PINN_beam_flow_TF2(..., weights={'w_fu': 5.0, 'w_wall': 20.0})
    """

    def __init__(self, Collo, INLET, OUTLET, WALL,
                 uv_layers, lb, ub,
                 rho=1.0, mu=0.02, weights=None):
        self.lb = lb.astype(np.float32).reshape(1, 2)
        self.ub = ub.astype(np.float32).reshape(1, 2)

        self.rho = tf.constant(rho, dtype=tf.float32)
        self.mu  = tf.constant(mu,  dtype=tf.float32)

        # Merge user-supplied weights with defaults
        w = dict(DEFAULT_WEIGHTS)
        if weights is not None:
            w.update(weights)
        self.w = {k: tf.constant(float(v), dtype=tf.float32) for k, v in w.items()}
        self._weights_dict = w   # keep a plain-Python copy for inspection / saving

        self.Collo  = Collo.astype(np.float32)
        self.INLET  = INLET.astype(np.float32)   # columns: x, y, u_ref, v_ref
        self.OUTLET = OUTLET.astype(np.float32)  # columns: x, y
        self.WALL   = WALL.astype(np.float32)    # columns: x, y  (no-slip)

        hidden_widths = uv_layers[1:-1]
        out_dim       = uv_layers[-1]  # must be 5
        self.net = MLP(hidden_widths + [out_dim], activation='tanh')

        # ── Loss history — all lists initialised here, never reset mid-run ──
        # Unweighted raw MSE for each term (useful for diagnosing convergence)
        self.loss_fu     = []   # MSE(f_u)
        self.loss_fv     = []   # MSE(f_v)
        self.loss_fs11   = []   # MSE(f_s11)
        self.loss_fs22   = []   # MSE(f_s22)
        self.loss_fs12   = []   # MSE(f_s12)
        self.loss_fp     = []   # MSE(f_p)
        self.loss_pde    = []   # unweighted sum: Σ MSE(f_i)
        self.loss_wall   = []   # unweighted MSE(u_wall, v_wall)
        self.loss_inlet  = []   # unweighted MSE(u_in − u_ref, v_in − v_ref)
        self.loss_outlet = []   # unweighted (mean p_out)²
        self.loss_total  = []   # weighted total (what the optimiser minimises)

    # ----------------------------------------------------------
    def net_psips(self, x, y):
        """Raw network forward pass.  Returns (ψ, p, s11, s22, s12)."""
        xy  = tf.concat([x, y], axis=1)
        out = self.net(xy)
        return out[:, 0:1], out[:, 1:2], out[:, 2:3], out[:, 3:4], out[:, 4:5]

    # ----------------------------------------------------------
    def uv_and_grads(self, x, y):
        """Compute (u, v) and all first-order derivatives needed for the PDE."""
        with tf.GradientTape(persistent=True) as tape:
            tape.watch([x, y])
            psi, p, s11, s22, s12 = self.net_psips(x, y)
            u =  tape.gradient(psi, y)
            v = -tape.gradient(psi, x)
        u_x  = tape.gradient(u,   x);  u_y  = tape.gradient(u,   y)
        v_x  = tape.gradient(v,   x);  v_y  = tape.gradient(v,   y)
        s11_x = tape.gradient(s11, x)
        s12_x = tape.gradient(s12, x);  s12_y = tape.gradient(s12, y)
        s22_y = tape.gradient(s22, y)
        del tape
        return (u, v, p, s11, s22, s12,
                u_x, u_y, v_x, v_y,
                s11_x, s12_x, s12_y, s22_y)

    # ----------------------------------------------------------
    def residuals(self, x, y):
        """Six PDE residuals for the mixed-form steady N-S equations."""
        rho, mu = self.rho, self.mu
        (u, v, p, s11, s22, s12,
         u_x, u_y, v_x, v_y,
         s11_x, s12_x, s12_y, s22_y) = self.uv_and_grads(x, y)

        f_u   = rho * (u * u_x + v * u_y) - s11_x - s12_y
        f_v   = rho * (u * v_x + v * v_y) - s12_x - s22_y
        f_s11 = -p + 2.0 * mu * u_x - s11
        f_s22 = -p + 2.0 * mu * v_y - s22
        f_s12 = mu * (u_y + v_x) - s12
        f_p   = p + 0.5 * (s11 + s22)
        return f_u, f_v, f_s11, f_s22, f_s12, f_p

    # ----------------------------------------------------------
    def loss_fn(self, Xc, WALL, INLET, OUTLET):
        """Composite loss with per-term weights.

        Returns
        -------
        total : weighted scalar loss (optimiser target)
        lfu, lfv, lfs11, lfs22, lfs12, lfp : unweighted MSE of each PDE residual
        lwall, lin, lout : unweighted MSE of each boundary condition
        """
        mse = lambda t: tf.reduce_mean(tf.square(t))
        w   = self.w

        # ── PDE residuals ──────────────────────────────────────
        xc, yc = Xc[:, 0:1], Xc[:, 1:2]
        fu, fv, fs11, fs22, fs12, fp = self.residuals(xc, yc)
        lfu   = mse(fu);   lfv   = mse(fv)
        lfs11 = mse(fs11); lfs22 = mse(fs22)
        lfs12 = mse(fs12); lfp   = mse(fp)

        loss_pde = (w['w_fu']  * lfu  + w['w_fv']   * lfv  +
                    w['w_fs11']* lfs11+ w['w_fs22']  * lfs22+
                    w['w_fs12']* lfs12+ w['w_fp']    * lfp)

        # ── Wall no-slip ────────────────────────────────────────
        xw, yw = WALL[:, 0:1], WALL[:, 1:2]
        u_w, v_w, *_ = self.uv_and_grads(xw, yw)
        lwall = mse(u_w) + mse(v_w)

        # ── Inlet velocity ──────────────────────────────────────
        xi, yi = INLET[:, 0:1], INLET[:, 1:2]
        ui, vi = INLET[:, 2:3], INLET[:, 3:4]
        u_i, v_i, *_ = self.uv_and_grads(xi, yi)
        lin = mse(u_i - ui) + mse(v_i - vi)

        # ── Outlet pressure ─────────────────────────────────────
        xo, yo = OUTLET[:, 0:1], OUTLET[:, 1:2]
        _, p_o, _, _, _ = self.net_psips(xo, yo)
        lout = tf.square(tf.reduce_mean(p_o))

        total = loss_pde + w['w_wall']*lwall + w['w_inlet']*lin + w['w_outlet']*lout
        return total, lfu, lfv, lfs11, lfs22, lfs12, lfp, lwall, lin, lout

    # ----------------------------------------------------------
    def _to_tf(self):
        """Convert stored numpy arrays to TF tensors once."""
        return (tf.constant(self.Collo,  dtype=tf.float32),
                tf.constant(self.WALL,   dtype=tf.float32),
                tf.constant(self.INLET,  dtype=tf.float32),
                tf.constant(self.OUTLET, dtype=tf.float32))

    def _record(self, total, lfu, lfv, lfs11, lfs22, lfs12, lfp, lwall, lin, lout):
        self.loss_total.append(float(total))
        self.loss_fu.append(float(lfu));     self.loss_fv.append(float(lfv))
        self.loss_fs11.append(float(lfs11)); self.loss_fs22.append(float(lfs22))
        self.loss_fs12.append(float(lfs12)); self.loss_fp.append(float(lfp))
        self.loss_pde.append(float(lfu) + float(lfv) + float(lfs11) +
                             float(lfs22) + float(lfs12) + float(lfp))
        self.loss_wall.append(float(lwall))
        self.loss_inlet.append(float(lin))
        self.loss_outlet.append(float(lout))

    # ----------------------------------------------------------
    def print_weights(self):
        """Print the current loss weights in a readable table."""
        print("Loss weights:")
        print(f"  PDE  — w_fu={self._weights_dict['w_fu']:.2g}  "
              f"w_fv={self._weights_dict['w_fv']:.2g}  "
              f"w_fs11={self._weights_dict['w_fs11']:.2g}  "
              f"w_fs22={self._weights_dict['w_fs22']:.2g}  "
              f"w_fs12={self._weights_dict['w_fs12']:.2g}  "
              f"w_fp={self._weights_dict['w_fp']:.2g}")
        print(f"  BC   — w_wall={self._weights_dict['w_wall']:.2g}  "
              f"w_inlet={self._weights_dict['w_inlet']:.2g}  "
              f"w_outlet={self._weights_dict['w_outlet']:.2g}")

    # ----------------------------------------------------------
    def train_adam(self, iters=10000, lr=5e-4, print_every=200):
        """Adam optimiser loop."""
        opt = tf.keras.optimizers.Adam(learning_rate=lr)
        Xc, WALL, INLET, OUTLET = self._to_tf()

        for it in range(iters):
            with tf.GradientTape() as tape:
                ret = self.loss_fn(Xc, WALL, INLET, OUTLET)
            total = ret[0]
            grads = tape.gradient(total, self.net.trainable_variables)
            opt.apply_gradients(zip(grads, self.net.trainable_variables))
            self._record(*[v.numpy() for v in ret])
            if it % print_every == 0:
                lfu, lfv, lfs11, lfs22, lfs12, lfp, lwall, lin, lout = \
                    [v.numpy() for v in ret[1:]]
                print(f"Adam {it:6d}/{iters} | total={total.numpy():.3e} | "
                      f"fu={lfu:.2e} fv={lfv:.2e} "
                      f"fs11={lfs11:.2e} fs22={lfs22:.2e} "
                      f"fs12={lfs12:.2e} fp={lfp:.2e} | "
                      f"wall={lwall:.2e} in={lin:.2e} out={lout:.2e}")

    # ----------------------------------------------------------
    def train_lbfgs(self, maxiter=50000, maxcor=50):
        """L-BFGS-B refinement via scipy.optimize.minimize."""
        Xc, WALL, INLET, OUTLET = self._to_tf()

        def pack():
            return np.concatenate(
                [v.numpy().ravel() for v in self.net.trainable_variables]
            ).astype(np.float64)

        def unpack(flat):
            idx = 0
            for v in self.net.trainable_variables:
                n = v.numpy().size
                v.assign(flat[idx:idx + n].reshape(v.shape).astype(np.float32))
                idx += n

        def loss_and_grad(flat):
            unpack(flat)
            with tf.GradientTape() as tape:
                ret = self.loss_fn(Xc, WALL, INLET, OUTLET)
            total = ret[0]
            grads = tape.gradient(total, self.net.trainable_variables)
            g_flat = np.concatenate(
                [g.numpy().ravel() for g in grads]
            ).astype(np.float64)
            return float(total.numpy()), g_flat

        def callback(xk):
            ret = self.loss_fn(Xc, WALL, INLET, OUTLET)
            self._record(*[v.numpy() for v in ret])
            total, lfu, lfv = ret[0], ret[1], ret[2]
            print(f"  L-BFGS | total={float(total):.3e} "
                  f"fu={float(lfu):.2e} fv={float(lfv):.2e}")

        res = scipy.optimize.minimize(
            fun=loss_and_grad, x0=pack(), jac=True,
            method='L-BFGS-B', callback=callback,
            options={'maxiter': maxiter, 'maxcor': maxcor,
                     'ftol': np.finfo(float).eps}
        )
        unpack(res.x)
        return res

    # ----------------------------------------------------------
    def save(self, path='uvNN_beam_weights'):
        self.net.save_weights(path)

    def load(self, path='uvNN_beam_weights'):
        self.net(tf.zeros((1, 2), dtype=tf.float32))  # build first
        self.net.load_weights(path)

    # ----------------------------------------------------------
    def predict(self, x_star, y_star):
        """Return (u, v, p) numpy arrays."""
        x = tf.constant(x_star.astype(np.float32))
        y = tf.constant(y_star.astype(np.float32))
        with tf.GradientTape(persistent=True) as tape:
            tape.watch([x, y])
            psi, p, _, _, _ = self.net_psips(x, y)
        u =  tape.gradient(psi, y)
        v = -tape.gradient(psi, x)
        del tape
        return u.numpy(), v.numpy(), p.numpy()

    def predict_all(self, x_star, y_star):
        """Return (u, v, p, s11, s22, s12) numpy arrays."""
        x = tf.constant(x_star.astype(np.float32))
        y = tf.constant(y_star.astype(np.float32))
        with tf.GradientTape(persistent=True) as tape:
            tape.watch([x, y])
            psi, p, s11, s22, s12 = self.net_psips(x, y)
        u =  tape.gradient(psi, y)
        v = -tape.gradient(psi, x)
        del tape
        return (u.numpy(), v.numpy(), p.numpy(),
                s11.numpy(), s22.numpy(), s12.numpy())

## 4. Loss Visualisation

In [ ]:
def plot_loss_all(model, n_adam=0):
    """Two-panel plot: PDE residuals (left) and BC residuals (right)."""
    iters = np.arange(len(model.loss_total))
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.semilogy(iters, model.loss_total,  'k-',   lw=2.0, label='Total (weighted)')
    ax1.semilogy(iters, model.loss_fu,     'b-',   lw=1.2, label=r'$f_u$ momentum x')
    ax1.semilogy(iters, model.loss_fv,     'r-',   lw=1.2, label=r'$f_v$ momentum y')
    ax1.semilogy(iters, model.loss_fs11,   'b--',  lw=1.0, label=r'$f_{s_{11}}$ constitutive')
    ax1.semilogy(iters, model.loss_fs22,   'r--',  lw=1.0, label=r'$f_{s_{22}}$ constitutive')
    ax1.semilogy(iters, model.loss_fs12,   'm--',  lw=1.0, label=r'$f_{s_{12}}$ shear')
    ax1.semilogy(iters, model.loss_fp,     'g--',  lw=1.0, label=r'$f_p$ trace/pressure')
    if 0 < n_adam < len(iters):
        ax1.axvline(n_adam, color='grey', ls='--', lw=1.2, label='Adam→L-BFGS')
    ax1.set_xlabel('Iteration'); ax1.set_ylabel('Unweighted MSE')
    ax1.set_title('PDE Residuals')
    ax1.legend(fontsize=8); ax1.grid(True, which='both', ls='--', alpha=0.35)

    ax2.semilogy(iters, model.loss_total,  'k-',   lw=2.0, label='Total (weighted)')
    ax2.semilogy(iters, model.loss_wall,   'g-',   lw=1.2, label='Wall (no-slip)')
    ax2.semilogy(iters, model.loss_inlet,  'b-',   lw=1.2, label='Inlet (velocity)')
    ax2.semilogy(iters, model.loss_outlet, 'r-',   lw=1.2, label='Outlet (pressure)')
    if 0 < n_adam < len(iters):
        ax2.axvline(n_adam, color='grey', ls='--', lw=1.2, label='Adam→L-BFGS')
    ax2.set_xlabel('Iteration'); ax2.set_ylabel('Unweighted MSE')
    ax2.set_title('Boundary Condition Residuals')
    ax2.legend(fontsize=8); ax2.grid(True, which='both', ls='--', alpha=0.35)

    fig.suptitle('Loss Convergence — All Components', fontsize=12)
    plt.tight_layout(); plt.show()

## Weighting Strategy Guide

Choosing good loss weights is a key tuning knob for PINN convergence.
The total loss is:

$$\mathcal{L}_{\text{total}} = w_{f_u}\mathcal{L}_{f_u} + w_{f_v}\mathcal{L}_{f_v} + w_{f_{s_{11}}}\mathcal{L}_{f_{s_{11}}} + w_{f_{s_{22}}}\mathcal{L}_{f_{s_{22}}} + w_{f_{s_{12}}}\mathcal{L}_{f_{s_{12}}} + w_{f_p}\mathcal{L}_{f_p} + w_{\text{wall}}\mathcal{L}_{\text{wall}} + w_{\text{in}}\mathcal{L}_{\text{in}} + w_{\text{out}}\mathcal{L}_{\text{out}}$$

### Strategy 1 — Uniform (baseline)
All weights = 1.  Simple starting point; rarely optimal.
```python
weights = {}  # uses DEFAULT_WEIGHTS: PDE=1, BC=10
```

### Strategy 2 — Momentum-Heavy
Momentum equations (`f_u`, `f_v`) involve nonlinear convection and stress divergence — they are harder to satisfy.  
Constitutive relations (`f_s11`, `f_s22`, `f_s12`) are purely algebraic and converge faster.
```python
weights = {'w_fu': 5.0, 'w_fv': 5.0,
           'w_fs11': 0.5, 'w_fs22': 0.5, 'w_fs12': 0.5,
           'w_fp': 1.0, 'w_wall': 10.0, 'w_inlet': 10.0, 'w_outlet': 5.0}
```

### Strategy 3 — BC-Dominant (physical initialisation)
Set BC weights very high early to force the network to honour boundary conditions first (acts like enforcing Dirichlet BCs strongly before solving the interior).  
Gradually reduce BC weight during training.
```python
weights_phase1 = {'w_wall': 100.0, 'w_inlet': 100.0, 'w_outlet': 50.0}
# train Adam with phase1, then rebuild with lower BC weights for L-BFGS
weights_phase2 = {'w_wall': 10.0,  'w_inlet': 10.0,  'w_outlet': 5.0}
```

### Strategy 4 — Magnitude Normalisation (recommended starting point)
Scale each weight by `1 / loss_i(0)` so all components start equal.  
Run a single forward pass at init, compute raw losses, then set weights.
```python
def auto_weights(model):
    Xc, WALL, INLET, OUTLET = model._to_tf()
    ret   = model.loss_fn(Xc, WALL, INLET, OUTLET)
    vals  = [float(v.numpy()) for v in ret[1:]]   # 9 unweighted MSEs
    keys  = ['w_fu','w_fv','w_fs11','w_fs22','w_fs12','w_fp',
             'w_wall','w_inlet','w_outlet']
    # normalise so each term contributes equally at step 0
    ref   = max(vals)
    return {k: ref / max(v, 1e-12) for k, v in zip(keys, vals)}
```

### Strategy 5 — Adaptive SoftAdapt (advanced)
Dynamically update weights based on the *rate of change* of each loss.  
Terms that are stagnating get higher weight; fast-converging terms get lower weight.
```python
def softadapt_weights(loss_hist, beta=0.1):
    # loss_hist: dict of lists (one entry per term)
    # Approximate instantaneous gradient of each loss
    rates = {}
    for k, v in loss_hist.items():
        if len(v) >= 2:
            rates[k] = abs(v[-1] - v[-2]) / (abs(v[-2]) + 1e-12)
        else:
            rates[k] = 1.0
    total_rate = sum(rates.values()) + 1e-12
    # Higher rate → easier to reduce → lower weight (let harder terms dominate)
    return {k: (1.0 - r / total_rate) for k, r in rates.items()}
```

### Strategy 6 — Gradient Normalisation (GradNorm-style)
Balance the **gradient magnitudes** of each loss term rather than the loss values.  
Ensures no single term dominates the parameter update direction.
This requires computing per-term gradients (expensive) — recommended only for final fine-tuning.


## 5. Weighting Configuration

Edit the `weights` dict below before running training.
Comment / uncomment the strategy you want to try.

In [ ]:
# ── Strategy 1: Uniform (all PDE=1, BC=10) ──────────────────────
# weights = {}   # uses DEFAULT_WEIGHTS

# ── Strategy 2: Momentum-Heavy ───────────────────────────────────
weights = {
    'w_fu':     5.0,   # momentum x — harder, boost it
    'w_fv':     5.0,   # momentum y
    'w_fs11':   0.5,   # constitutive — algebraic, relax
    'w_fs22':   0.5,
    'w_fs12':   0.5,
    'w_fp':     1.0,   # trace condition
    'w_wall':  10.0,   # no-slip (cylinder + beam + channel)
    'w_inlet': 10.0,   # prescribed inlet velocity
    'w_outlet': 5.0,   # outlet pressure
}

# ── Strategy 3: BC-Dominant (phase 1 / warm-up) ──────────────────
# weights_phase1 = {'w_wall': 100.0, 'w_inlet': 100.0, 'w_outlet': 50.0}
# weights_phase2 = {'w_wall': 10.0,  'w_inlet': 10.0,  'w_outlet': 5.0}

# ── Strategy 4: Magnitude Normalisation (auto-calibrate at init) ──
# def auto_weights(model):
#     Xc, WALL, INLET, OUTLET = model._to_tf()
#     ret  = model.loss_fn(Xc, WALL, INLET, OUTLET)
#     vals = [float(v.numpy()) for v in ret[1:]]
#     keys = ['w_fu','w_fv','w_fs11','w_fs22','w_fs12','w_fp',
#             'w_wall','w_inlet','w_outlet']
#     ref  = max(vals)
#     return {k: ref / max(v, 1e-12) for k, v in zip(keys, vals)}

print('Weights configured.')

## 6. Problem Setup — Turek-Hron Geometry

| Parameter | Value | Notes |
|-----------|-------|-------|
| Channel   | 1.1 × 0.41 | |
| Cylinder  | center (0.2, 0.2), r = 0.05 | d = 0.1 |
| Beam      | l = 0.35, h = 0.02 | |
| Beam box  | x ∈ [0.25, 0.60], y ∈ [0.19, 0.21] | |
| ρ, ν      | 1.0, 0.02 → μ = 0.02 | CFD1: Re = Ū·d/ν ≈ 20 |
| Ū, U_max  | 0.2, 0.3 m/s | parabolic inlet |

In [ ]:
L, H = 1.1, 0.41
lb   = np.array([0.0, 0.0])
ub   = np.array([L, H])
xc, yc, r = 0.2, 0.2, 0.05

l_beam, h_beam = 0.35, 0.02
x_right, y_bot = 0.6, 0.19
x_left  = x_right - l_beam    # 0.25
y_top   = y_bot   + h_beam    # 0.21

rho  = 1.0; nu = 0.02; mu = rho * nu
Ubar = 0.2; U_max = 1.5 * Ubar

Nw, Ni, No, Nc, Nb = 441, 201, 201, 400, 300
wall_up = np.hstack([L*lhs(1,Nw), H*np.ones((Nw,1))])
wall_lw = np.hstack([L*lhs(1,Nw), np.zeros((Nw,1))])

y_in  = H * lhs(1, Ni)
u_in  = 4*U_max*y_in*(H-y_in)/H**2
INLET = np.hstack([np.zeros((Ni,1)), y_in, u_in, np.zeros_like(u_in)])
OUTLET = np.hstack([L*np.ones((No,1)), H*lhs(1,No)])

theta_c = 2*np.pi*lhs(1, Nc)
CYLD    = np.hstack([xc+r*np.cos(theta_c), yc+r*np.sin(theta_c)])

beam_top   = np.hstack([x_left+(x_right-x_left)*lhs(1,Nb), y_top*np.ones((Nb,1))])
beam_bot   = np.hstack([x_left+(x_right-x_left)*lhs(1,Nb), y_bot*np.ones((Nb,1))])
beam_right = np.hstack([x_right*np.ones((Nb,1)), y_bot+(y_top-y_bot)*lhs(1,Nb)])
BEAM = np.vstack([beam_top, beam_bot, beam_right])
WALL = np.vstack([CYLD, BEAM, wall_up, wall_lw])

XY_c      = lb + (ub-lb) * lhs(2, 4000)
XY_refine = np.array([0.1,0.1]) + np.array([0.6,0.2]) * lhs(2, 1000)
XY_c      = np.vstack([XY_c, XY_refine])
XY_c      = DelObsPT(XY_c, xc, yc, r, x_left, x_right, y_bot, y_top)
XY_c      = np.vstack([XY_c, WALL, OUTLET, INLET[:,0:2]])
print('Collocation points:', XY_c.shape[0])

fig, ax = plt.subplots(figsize=(10, 2.5))
ax.set_aspect('equal')
ax.scatter(XY_c[:,0], XY_c[:,1], s=0.5, alpha=0.08)
ax.scatter(WALL[:,0], WALL[:,1], s=5, alpha=0.6)
ax.set_title('Collocation & boundary points (Cylinder + Beam)')
plt.tight_layout(); plt.show()

## 7. Create Model & Print Weights

In [ ]:
uv_layers = [2] + 8*[40] + [5]

model = PINN_beam_flow_TF2(
    XY_c, INLET, OUTLET, WALL, uv_layers, lb, ub,
    rho=rho, mu=mu, weights=weights
)
model.print_weights()

## 8. Training — Adam + L-BFGS-B

In [ ]:
t0 = time.time()
model.train_adam(iters=10000, lr=5e-4, print_every=200)
n_adam = len(model.loss_total)
res    = model.train_lbfgs(maxiter=50000)
print(f'Total training time: {time.time()-t0:.1f} s')

model.save('uvNN_tf2_weights_beam1_weight10.weights.h5')

## 9. Loss Convergence — All Components

In [ ]:
plot_loss_all(model, n_adam=n_adam)

### 9b. Snapshot table — final unweighted residuals

Lets you check whether any PDE equation is substantially worse than the others.

In [ ]:
labels = ['f_u','f_v','f_s11','f_s22','f_s12','f_p','wall','inlet','outlet']
hists  = [model.loss_fu, model.loss_fv, model.loss_fs11, model.loss_fs22,
          model.loss_fs12, model.loss_fp, model.loss_wall,
          model.loss_inlet, model.loss_outlet]

print(f'{'Term':>10}  {'Init MSE':>12}  {'Final MSE':>12}  {'Reduction':>12}')
print('-'*50)
for lbl, h in zip(labels, hists):
    if len(h) > 0:
        print(f'{lbl:>10}  {h[0]:12.4e}  {h[-1]:12.4e}  {h[0]/max(h[-1],1e-15):12.1f}×')

## 10. Field Visualisation (u, v, p)

In [ ]:
nx, ny = 251, 101
xg = np.linspace(0.0, L, nx); yg = np.linspace(0.0, H, ny)
Xg, Yg = np.meshgrid(xg, yg)
xs = Xg.ravel()[:,None]; ys = Yg.ravel()[:,None]

dst_cyl = np.sqrt((xs-xc)**2 + (ys-yc)**2)
in_beam = ((xs[:,0]>=x_left)&(xs[:,0]<=x_right)&
            (ys[:,0]>=y_bot) &(ys[:,0]<=y_top))
mask    = (dst_cyl[:,0]>=r) & ~in_beam
x_ev, y_ev = xs[mask], ys[mask]

u_p, v_p, p_p = model.predict(x_ev, y_ev)

theta_plot = np.linspace(0, 2*np.pi, 400)
cyl_x = xc+r*np.cos(theta_plot); cyl_y = yc+r*np.sin(theta_plot)
beam_px = [x_left,x_right,x_right,x_left,x_left]
beam_py = [y_bot, y_bot, y_top, y_top, y_bot]

fig, axes = plt.subplots(3, 1, figsize=(11, 9), constrained_layout=True)
for axh, field, label in zip(axes,
        [u_p.ravel(), v_p.ravel(), p_p.ravel()],
        [r'$u^*$', r'$v^*$', r'$p^*$']):
    sc = axh.scatter(x_ev[:,0], y_ev[:,0], c=field, s=2, cmap='jet')
    axh.plot(cyl_x, cyl_y, 'k-', lw=1.2)
    axh.plot(beam_px, beam_py, 'k-', lw=1.2)
    axh.set_aspect('equal'); axh.axis('off')
    axh.set_title(label + ' (PINN)', fontsize=11)
    plt.colorbar(sc, ax=axh, fraction=0.02, pad=0.01)
plt.show()

## 11. Pressure Distribution on Cylinder Surface

In [ ]:
theta_cyl = np.linspace(0, 2*np.pi, 361)
xc_pts = xc + r*np.cos(theta_cyl)
yc_pts = yc + r*np.sin(theta_cyl)
_, _, p_cyl = model.predict(xc_pts[:,None], yc_pts[:,None])

fig, ax = plt.subplots(figsize=(8, 4.5), dpi=120)
ax.plot(theta_cyl, p_cyl.ravel(), lw=2.5, label='PINN')
ax.set_xlim(0, 2*np.pi)
ax.set_xlabel(r'$\theta$ (rad)', fontsize=14)
ax.set_ylabel('Pressure', fontsize=14)
ax.set_xticks([0, np.pi/2, np.pi, 3*np.pi/2, 2*np.pi])
ax.set_xticklabels([r'$0$', r'$\pi/2$', r'$\pi$', r'$3\pi/2$', r'$2\pi$'])
ax.legend(fontsize=12, frameon=False); ax.grid(alpha=0.4)
ax.set_title('Pressure on cylinder surface')
plt.tight_layout(); plt.show()

## 12. Drag & Lift on Cylinder + Beam

In [ ]:
import numpy as np

def _traction_from_ps(model, x, y, nx, ny):
    """
    Traction t = σ n where σ components are directly predicted:
      σ = [[s11, s12],
           [s12, s22]]
    """
    x = np.asarray(x).reshape(-1, 1).astype(np.float32)
    y = np.asarray(y).reshape(-1, 1).astype(np.float32)
    nx = np.asarray(nx).reshape(-1)
    ny = np.asarray(ny).reshape(-1)

    # predict (u, v, p, s11, s22, s12)
    _, _, _, s11, s22, s12 = model.predict_all(x, y)

    s11 = np.asarray(s11).reshape(-1)
    s22 = np.asarray(s22).reshape(-1)
    s12 = np.asarray(s12).reshape(-1)

    tx = s11*nx + s12*ny
    ty = s12*nx + s22*ny
    return tx, ty


def compute_drag_lift_cylinder_beam(
    model,
    # cylinder
    xc=0.2, yc=0.2, r=0.05,
    # beam (Turek–Hron default: length=0.35, thickness=0.02)
    beam_L=0.35, beam_h=0.02,
    # quadrature resolution
    n_theta=2000,
    n_edge=800,
    # nondimensional coefficients (optional)
    nondim=True,
    rho=1000.0,
    Ubar=0.2,
    d=None,
):
    """
    Drag/Lift on combined obstacle: cylinder + attached rigid beam.

    Beam is assumed attached at cylinder rightmost point:
        x from x_attach = xc + r  to x_end = x_attach + beam_L
        y in [yc - beam_h/2, yc + beam_h/2]

    Boundary integrated:
      - cylinder arc excluding attachment window (covered by beam)
      - beam top edge, bottom edge, right edge
      - (beam left edge NOT included; it is inside the solid attachment)

    Returns:
      if nondim: (D, L, Cd, Cl)
      else:      (D, L)
    """
    if d is None:
        d = 2.0 * r

    # ---------- 1) Cylinder arc (exclude attachment zone) ----------
    # attachment window on cylinder: y in [yc - beam_h/2, yc + beam_h/2] at x≈xc+r
    # => theta0 = arcsin((beam_h/2)/r)
    theta0 = np.arcsin(np.clip((beam_h * 0.5) / r, 0.0, 1.0))

    # sample theta in (theta0, 2π-theta0)
    th = np.linspace(theta0, 2*np.pi - theta0, n_theta, endpoint=False)
    x_c = xc + r*np.cos(th)
    y_c = yc + r*np.sin(th)
    nx_c = np.cos(th)
    ny_c = np.sin(th)
    dtheta = (2*np.pi - 2*theta0) / n_theta
    dS_c = r * dtheta

    tx_c, ty_c = _traction_from_ps(model, x_c, y_c, nx_c, ny_c)
    D_c = np.sum(tx_c) * dS_c
    L_c = np.sum(ty_c) * dS_c

    # ---------- 2) Beam exposed edges ----------
    x_attach = xc + r
    x_end    = x_attach + beam_L
    y_top    = yc + 0.5*beam_h
    y_bot    = yc - 0.5*beam_h

    # Top edge: (x_attach -> x_end), y=y_top, n=(0, +1)
    xs_top = np.linspace(x_attach, x_end, n_edge, endpoint=False)
    ys_top = np.full_like(xs_top, y_top)
    nx_top = np.zeros_like(xs_top)
    ny_top = np.ones_like(xs_top)
    ds_top = (x_end - x_attach) / n_edge

    tx_top, ty_top = _traction_from_ps(model, xs_top, ys_top, nx_top, ny_top)
    D_top = np.sum(tx_top) * ds_top
    L_top = np.sum(ty_top) * ds_top

    # Bottom edge: n=(0, -1)
    xs_bot = np.linspace(x_attach, x_end, n_edge, endpoint=False)
    ys_bot = np.full_like(xs_bot, y_bot)
    nx_bot = np.zeros_like(xs_bot)
    ny_bot = -np.ones_like(xs_bot)
    ds_bot = (x_end - x_attach) / n_edge

    tx_bot, ty_bot = _traction_from_ps(model, xs_bot, ys_bot, nx_bot, ny_bot)
    D_bot = np.sum(tx_bot) * ds_bot
    L_bot = np.sum(ty_bot) * ds_bot

    # Right edge: x=x_end, y_bot -> y_top, n=(+1, 0)
    ys_right = np.linspace(y_bot, y_top, n_edge, endpoint=False)
    xs_right = np.full_like(ys_right, x_end)
    nx_right = np.ones_like(ys_right)
    ny_right = np.zeros_like(ys_right)
    ds_right = (y_top - y_bot) / n_edge

    tx_r, ty_r = _traction_from_ps(model, xs_right, ys_right, nx_right, ny_right)
    D_r = np.sum(tx_r) * ds_right
    L_r = np.sum(ty_r) * ds_right

    # Total
    D = D_c + D_top + D_bot + D_r
    L = L_c + L_top + L_bot + L_r

    if not nondim:
        return D, L

    q = 0.5 * rho * (Ubar**2) * d
    Cd = D / q
    Cl = L / q
    return D, L, Cd, Cl


# -----------------------
# Example (CFD1)
# -----------------------
rho = 1.0
Ubar = 0.2
D, L, Cd, Cl = compute_drag_lift_cylinder_beam(
    model,
    xc=0.2, yc=0.2, r=0.05,
    beam_L=0.35, beam_h=0.02,   # Turek–Hron beam
    n_theta=3000,
    n_edge=1200,
    nondim=True,
    rho=rho,
    Ubar=Ubar,
    d=0.1
)
print(f"D={D:.6e}, L={L:.6e}  (2D force per unit depth)")
print(f"Cd={Cd:.6f}, Cl={Cl:.6f}")
print(f"Cd_{True}={0.2}, Cl_{True}={4e-3}")


D, L_f, Cd, Cl = compute_drag_lift_cylinder_beam(
    model, xc=xc, yc=yc, r=r,
    beam_L=l_beam, beam_h=h_beam,
    rho=rho, Ubar=Ubar
)
print(f'Drag D = {D:.6f}   Lift L = {L_f:.6f}')
print(f'Cd = {Cd:.4f}   Cl = {Cl:.4f}')

D, L_f, Cd, Cl = compute_drag_lift_cylinder_beam(
    model, xc=xc, yc=yc, r=r,
    beam_L=l_beam, beam_h=h_beam,
    rho=rho, Ubar=Ubar
)
print(f'Drag D = {D:.6f}   Lift L = {L_f:.6f}')
print(f'Cd = {Cd:.4f}   Cl = {Cl:.4f}')
print('Turek-Hron CFD1 reference: Cd ≈ 5.58, Cl ≈ 0.0107')

## 13. Save Results

In [ ]:
results = {
    'loss_total':  model.loss_total,
    'loss_fu':     model.loss_fu,    'loss_fv':    model.loss_fv,
    'loss_fs11':   model.loss_fs11,  'loss_fs22':  model.loss_fs22,
    'loss_fs12':   model.loss_fs12,  'loss_fp':    model.loss_fp,
    'loss_pde':    model.loss_pde,
    'loss_wall':   model.loss_wall,
    'loss_inlet':  model.loss_inlet,
    'loss_outlet': model.loss_outlet,
    'u_pred': u_p, 'v_pred': v_p, 'p_pred': p_p,
    'x_eval': x_ev, 'y_eval': y_ev,
    'drag': D, 'lift': L_f, 'Cd': Cd, 'Cl': Cl,
    'meta': {
        'rho': rho, 'mu': mu, 'Ubar': Ubar,
        'weights': model._weights_dict,
        'layers': uv_layers,
        'description': 'Steady cylinder+beam PINN (Turek-Hron CFD1)'
    }
}
with open('pinn_cylinder_beam1_results_weight10.pkl', 'wb') as f:
    pickle.dump(results, f)
print('Results saved.')